# 01 - Exploratory Data Analysis

**Project:** Healthcare Readmission Risk Pipeline
**Layer:** Gold - mart_patient_features

This notebook covers:
- Dataset loading and basic statistics
- Distribution analysis of key variables
- Class balance assessment
- Correlation structure and bivariate analysis
- Dimension-level readmission rates

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# Load feature store from PostgreSQL or CSV
# Option A: PostgreSQL
# import psycopg2
# conn = psycopg2.connect(dbname='healthcare_dw', user='postgres', password='postgres')
# df = pd.read_sql('SELECT * FROM gold.mart_patient_features', conn)

# Option B: CSV export
# df = pd.read_csv('data/mart_patient_features.csv')

print('Dataset loaded.')

## 1. Basic Statistics

In [ ]:
print('Shape:', df.shape)
print('\nClass balance:')
print(df['readmission_30j'].value_counts(normalize=True).round(3))
print('\nMissing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])
df.describe().round(2)

## 2. Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class balance
df['readmission_30j'].value_counts().plot(kind='bar', ax=axes[0], color=['#2196F3', '#F44336'])
axes[0].set_title('Readmission class balance')
axes[0].set_xlabel('Readmitted (0=No, 1=Yes)')
axes[0].set_ylabel('Count')
axes[0].tick_params(rotation=0)

# Readmission rate by pathology
df.groupby('pathologie')['readmission_30j'].mean().sort_values(ascending=True).plot(
    kind='barh', ax=axes[1], color='#FF9800')
axes[1].set_title('Readmission rate by pathology')
axes[1].set_xlabel('Rate')

plt.tight_layout()
plt.show()

## 3. Continuous Variable Distributions

In [ ]:
numerical_cols = ['age', 'nb_hospitalisations_precedentes', 'nb_medicaments',
                  'duree_sejour', 'indice_complexite', 'ratio_meds_duree']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flatten(), numerical_cols):
    for label, color in [(0, '#2196F3'), (1, '#F44336')]:
        df[df['readmission_30j'] == label][col].hist(
            bins=30, alpha=0.6, ax=ax, label=f'readmission={label}', color=color)
    ax.set_title(col)
    ax.legend()

plt.suptitle('Feature distributions by readmission status', y=1.02)
plt.tight_layout()
plt.show()

## 4. Correlation Matrix

In [ ]:
binary_and_numerical = numerical_cols + ['flag_polymedication', 'flag_sortie_complexe',
                                          'flag_antecedents_lourds', 'readmission_30j']

corr = df[binary_and_numerical].corr()

plt.figure(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, linewidths=0.5)
plt.title('Feature correlation matrix (lower triangle)')
plt.tight_layout()
plt.show()

## 5. Readmission Rate by Dimension

In [ ]:
dims = ['tranche_age', 'service', 'mode_sortie', 'groupe_clinique']
global_rate = df['readmission_30j'].mean()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ax, dim in zip(axes.flatten(), dims):
    rates = df.groupby(dim)['readmission_30j'].mean().sort_values(ascending=True)
    bars = rates.plot(kind='barh', ax=ax, color='#5C6BC0', alpha=0.8)
    ax.axvline(global_rate, color='red', linestyle='--', alpha=0.7, label=f'Global avg: {global_rate:.1%}')
    ax.set_title(f'Readmission rate by {dim}')
    ax.set_xlabel('Rate')
    ax.legend()
    for bar, val in zip(bars.patches, rates):
        ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
                f'{val:.1%}', va='center', fontsize=8)

plt.tight_layout()
plt.show()

## 6. Summary

In [ ]:
summary = pd.DataFrame({
    'n_stays': [len(df)],
    'readmission_rate': [df['readmission_30j'].mean().round(3)],
    'mean_age': [df['age'].mean().round(1)],
    'mean_los': [df['duree_sejour'].mean().round(1)],
    'mean_medications': [df['nb_medicaments'].mean().round(1)],
    'mean_prior_hosp': [df['nb_hospitalisations_precedentes'].mean().round(2)],
})

print('=== DATASET SUMMARY ===')
print(summary.T.to_string())